# Training a Mini-LLaMA from scratch in PyTorch 

In [5]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.append(str(project_root))
print(project_root)


C:\Users\3Dfils\Desktop\learning llms


In [13]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cpu


In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-2-7b-hf")

tokenizer.pad_token = tokenizer.eos_token

text = "Hello, I am training a mini llama."

tokens = tokenizer(text, return_tensors="pt")

print(tokens["input_ids"])
decoded = tokenizer.decode(tokens["input_ids"][0])

print(decoded)

tensor([[    1, 15043, 29892,   306,   626,  6694,   263, 20629, 11148,  3304,
         29889]])
<s> Hello, I am training a mini llama.


In [ ]:
from Models.llama import Llama
from Models.config import LLAMA_CONFIG

LLAMA_CONFIG["vocab_size"] = tokenizer.vocab_size

model = Llama(LLAMA_CONFIG)
model = model.to(device)
total_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_parameters}")

Total parameters: 38408704


In [10]:
from datasets import load_dataset

dataset = load_dataset("roneneldan/TinyStories")

text = dataset["train"][0]["text"]
print(text)

One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.


In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class LlamaDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.max_length = max_length
        self.stride = stride

        self.tokens = tokenizer.encode(txt, add_special_tokens=False)

    def __len__(self):
        return (len(self.tokens) - self.max_length) // self.stride + 1

    def __getitem__(self, idx):
        start = idx * self.stride

        x = self.tokens[start : start + self.max_length]

        y = self.tokens[start + 1 : start + self.max_length + 1]

        return (torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long))
    
train_data = LlamaDataset(
    " ".join(dataset["train"]["text"][:1000]),
    tokenizer,
    max_length=LLAMA_CONFIG["context_length"],
    stride=128
)

train_dataloader = DataLoader(
    train_data,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

val_data = LlamaDataset(
    " ".join(dataset["validation"]["text"][:1000]),
    tokenizer,
    max_length=LLAMA_CONFIG["context_length"],
    stride=128,
)

val_dataloader = DataLoader(
    val_data,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    drop_last=True,
)

x, y = next(iter(train_dataloader))

print(x.shape)
print(y.shape)
print(x.dtype)
print(y.dtype)

torch.Size([8, 256])
torch.Size([8, 256])
torch.int64
torch.int64


In [14]:
import torch.nn.functional as F

def calc_loss_batch(model,input,target,device):
    input = input.to(device)
    target = target.to(device)

    logits = model(input) # [batch,seq_len,vocab_size]

    loss = F.cross_entropy(
        logits.reshape(-1,logits.size(-1)),
        target.reshape(-1)
    )

    return loss

model.eval()

input, target = next(iter(train_dataloader))

loss = calc_loss_batch(model,input,target,device)

print(loss.item())


468.0416259765625


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)

In [ ]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0
    if len(data_loader) == 0:
        return torch.nan
    elif num_batches is None:
        num_batches = len(
            data_loader
        )  # If no number of batches specified, iterates over all
    else:
        num_batches = min(num_batches, len(data_loader))

    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        loss = calc_loss_batch(input_batch, target_batch, model, device)
        total_loss += loss.item()
    return total_loss / num_batches


In [ ]:
def evaluate_model(
    model, train_loader, val_loader, device, eval_iter
):
    model.eval()  # Puts the model in eval mode, disables dropouts
    with torch.no_grad():  # Disables gradient tracking
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        validation_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, validation_loss


In [ ]:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text)
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor


def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())


def generate_text(model, idx, max_new_tokens, context_size, temperature=1.0, top_k=50):
    # Generate tokens one by one
    for _ in range(max_new_tokens):
        # Keep only the last context_size tokens because the model cannot process sequences longer than its context window
        idx_cond = idx[:, -context_size:]

        # Disable gradient computation during inference
        with torch.no_grad():
            # Forward pass through the model logits shape: (batch_size, seq_len, vocab_size)
            logits = model(idx_cond)

        # Select logits from the last generated token We only care about predicting the next token Shape:(batch_size, vocab_size)
        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1].unsqueeze(-1)
            logits = torch.where(
                logits < min_val,
                torch.tensor(float("-inf"), device=logits.device),
                logits,
            )

        # Convert logits into probabilities
        probs = torch.softmax(logits, dim=-1)

        # Select the token with highest probability Shape: (batch_size, 1)
        idx_next = torch.multinomial(probs, num_samples=1)

        # Append the predicted token to the sequence Previous: (batch_size, seq_len) After concatenation:(batch_size, seq_len + 1)
        idx = torch.cat((idx, idx_next), dim=-1)

    # Return complete generated sequence
    return idx


def generate_sample(model, tokenizer, device, start_context):
    model.eval()
    context_size = LLAMA_CONFIG["context_length"]
    encoded = text_to_token_ids(start_context, tokenizer).to(device)

    with torch.no_grad():
        token_ids = generate_text(
            model=model,
            idx=encoded,
            max_new_tokens=50,
            context_size=context_size,
            temperature=0.8,
        )
    decoded_text = token_ids_to_text(token_ids, tokenizer)
    print(decoded_text.replace("\n", " "))
    model.train()


generate_sample(model, tokenizer, device, "Hello, I am training a mini")

<s> Hello, I am training a mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini mini


In [ ]:
def train(
    model,
    train_data_loader,
    val_data_loader,
    optimizer,
    device,
    epochs,
    eval_freq,
    eval_iter,
    start_context,
    tokenizer,
    save_weights=False,
    train_name="",
):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1



    for epoch in range(epochs):
        
        model.train()  # Puts the model in train mode
        for input_batch, target_batch in train_data_loader:
            optimizer.zero_grad()  # Reset loss gradients from previous batch train
            loss = calc_loss_batch(model, input_batch, target_batch, device)

            loss.backward()  # Calculate loss gradients
            optimizer.step()  # Update model weights

            tokens_seen += input_batch.numel()
            global_step += 1

            # Evaluation step
            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model,
                    train_data_loader,
                    val_data_loader,
                    device,
                    eval_iter,
                    
                )
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)

                print(
                    f"Epoch:{epoch + 1} (Step {global_step:06d}): Train loss {train_loss:.3f} Validation loss {val_loss:.3f}"
                )
        if save_weights:
            checkpoint = {
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_losses": train_losses,
                "val_losses": val_losses,
                "tokens_seen": tokens_seen,
                "track_tokens_seen": track_tokens_seen,
                "config": LLAMA_CONFIG,
                "train_name": train_name,
            }
            weights_dir = Path("weights")
            weights_dir.mkdir(parents=True, exist_ok=True)
            torch.save(
                checkpoint,
                weights_dir / f"{model.__class__.__name__}_{epoch}_{train_name}.pt",
            )

        generate_sample(model, tokenizer, device, start_context)
    return train_losses, val_losses, track_tokens_seen
